<a href="https://colab.research.google.com/github/Fatou-Kine3/github_task/blob/main/Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#3 Problem 1

In [1]:
import numpy as np
import keras
import os
from pathlib import Path

In [ ]:
print("Keras version:", keras.__version__)

Keras version: 3.13.2


In [ ]:
import tensorflow as tf
from pathlib import Path

fpath = tf.keras.utils.get_file(
    fname="fra-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
    extract=True,
)

print("Downloaded file:")
print(fpath)

print("\nFiles in the directory:")
for p in Path(fpath).parent.rglob("*"):
    print(p)

Downloaded file:
/root/.keras/datasets/fra-eng_extracted

Files in the directory:
/root/.keras/datasets/fra-eng_extracted
/root/.keras/datasets/fra-eng.zip
/root/.keras/datasets/fra-eng_extracted/_about.txt
/root/.keras/datasets/fra-eng_extracted/fra.txt


In [ ]:
from pathlib import Path

# Search for fra.txt
fra_files = list(Path("/root/.keras/datasets").rglob("fra.txt"))

print("Found files:")
for file in fra_files:
    print(file)

Found files:
/root/.keras/datasets/fra-eng_extracted/fra.txt


In [ ]:
from pathlib import Path

fra_files = list(Path("/root/.keras/datasets").rglob("fra.txt"))

if len(fra_files) == 0:
    raise FileNotFoundError("fra.txt was not found.")

data_path = fra_files[0]

print("Dataset found at:")
print(data_path)

Dataset found at:
/root/.keras/datasets/fra-eng_extracted/fra.txt


In [ ]:
print("File exists:", data_path.exists())

File exists: True


In [ ]:
with open(data_path, "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

Go.	Va !
Hi.	Salut !
Run!	Cours !
Run!	Courez !
Who?	Qui ?


In [ ]:
# ============================================================
# PROBLEM 1 — CHARACTER-LEVEL SEQ2SEQ MACHINE TRANSLATION
# ============================================================

import numpy as np
import keras
import os

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

batch_size = 64
epochs = 20              # Reduced from official 100 epochs
latent_dim = 256
num_samples = 10000

print("Keras version:", keras.__version__)
print("Training samples:", num_samples)
print("Epochs:", epochs)
print("Latent dimension:", latent_dim)


# ------------------------------------------------------------
# 2. CHECK DATASET
# ------------------------------------------------------------

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Dataset not found at: {data_path}\n"
        "Please make sure data_path points to fra.txt."
    )

print("\nDataset found:", data_path)


# ------------------------------------------------------------
# 3. READ AND PREPARE TEXT DATA
# ------------------------------------------------------------

input_texts = []
target_texts = []

input_characters = set()
target_characters = set()

with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")

for line in lines[: min(num_samples, len(lines) - 1)]:

    # Some lines may contain more than 3 fields because of
    # additional information. We only need the first 3.
    parts = line.split("\t")

    if len(parts) < 2:
        continue

    input_text = parts[0]
    target_text = parts[1]

    # Start-of-sequence = \t
    # End-of-sequence = \n
    target_text = "\t" + target_text + "\n"

    input_texts.append(input_text)
    target_texts.append(target_text)

    for char in input_text:
        input_characters.add(char)

    for char in target_text:
        target_characters.add(char)


# ------------------------------------------------------------
# 4. CREATE VOCABULARIES
# ------------------------------------------------------------

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))

num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)

max_encoder_seq_length = max(
    len(txt) for txt in input_texts
)

max_decoder_seq_length = max(
    len(txt) for txt in target_texts
)

print("\n===== DATA INFORMATION =====")
print("Number of samples:", len(input_texts))
print("Number of input characters:", num_encoder_tokens)
print("Number of target characters:", num_decoder_tokens)
print("Maximum encoder sequence length:", max_encoder_seq_length)
print("Maximum decoder sequence length:", max_decoder_seq_length)


# ------------------------------------------------------------
# 5. CHARACTER → INDEX DICTIONARIES
# ------------------------------------------------------------

input_token_index = {
    char: i for i, char in enumerate(input_characters)
}

target_token_index = {
    char: i for i, char in enumerate(target_characters)
}


# ------------------------------------------------------------
# 6. CREATE ONE-HOT ENCODED TENSORS
# ------------------------------------------------------------

encoder_input_data = np.zeros(
    (
        len(input_texts),
        max_encoder_seq_length,
        num_encoder_tokens
    ),
    dtype="float32"
)

decoder_input_data = np.zeros(
    (
        len(input_texts),
        max_decoder_seq_length,
        num_decoder_tokens
    ),
    dtype="float32"
)

decoder_target_data = np.zeros(
    (
        len(input_texts),
        max_decoder_seq_length,
        num_decoder_tokens
    ),
    dtype="float32"
)


# ------------------------------------------------------------
# 7. ONE-HOT ENCODING
# ------------------------------------------------------------

for i, (input_text, target_text) in enumerate(
    zip(input_texts, target_texts)
):

    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[
            i, t, input_token_index[char]
        ] = 1.0

    # Padding encoder input with spaces
    for t in range(len(input_text), max_encoder_seq_length):
        encoder_input_data[
            i, t, input_token_index[" "]
        ] = 1.0

    # Decoder input
    for t, char in enumerate(target_text):

        decoder_input_data[
            i, t, target_token_index[char]
        ] = 1.0

        # Decoder target is shifted one timestep forward
        if t > 0:
            decoder_target_data[
                i,
                t - 1,
                target_token_index[char]
            ] = 1.0

    # Padding decoder input
    for t in range(len(target_text), max_decoder_seq_length):
        decoder_input_data[
            i, t, target_token_index[" "]
        ] = 1.0

    # Padding decoder target
    for t in range(len(target_text) - 1, max_decoder_seq_length):
        decoder_target_data[
            i, t, target_token_index[" "]
        ] = 1.0


print("\n===== TENSOR SHAPES =====")
print("Encoder input:", encoder_input_data.shape)
print("Decoder input:", decoder_input_data.shape)
print("Decoder target:", decoder_target_data.shape)


# ------------------------------------------------------------
# 8. BUILD ENCODER
# ------------------------------------------------------------

encoder_inputs = keras.Input(
    shape=(None, num_encoder_tokens),
    name="encoder_inputs"
)

encoder_lstm = keras.layers.LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_inputs
)

encoder_states = [state_h, state_c]


# ------------------------------------------------------------
# 9. BUILD DECODER
# ------------------------------------------------------------

decoder_inputs = keras.Input(
    shape=(None, num_decoder_tokens),
    name="decoder_inputs"
)

decoder_lstm = keras.layers.LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_inputs,
    initial_state=encoder_states
)

decoder_dense = keras.layers.Dense(
    num_decoder_tokens,
    activation="softmax",
    name="decoder_dense"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)


# ------------------------------------------------------------
# 10. CREATE TRAINING MODEL
# ------------------------------------------------------------

model = keras.Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name="seq2seq_translation_model"
)

print("\n===== MODEL SUMMARY =====")
model.summary()


# ------------------------------------------------------------
# 11. COMPILE
# ------------------------------------------------------------

model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# ------------------------------------------------------------
# 12. TRAIN
# ------------------------------------------------------------

print("\n===== STARTING TRAINING =====")

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
    verbose=1
)


# ------------------------------------------------------------
# 13. SAVE TRAINED MODEL
# ------------------------------------------------------------

model.save("s2s_model.keras")

print("\nTraining model saved as:")
print("s2s_model.keras")


# ------------------------------------------------------------
# 14. BUILD ENCODER INFERENCE MODEL
# ------------------------------------------------------------

encoder_model = keras.Model(
    encoder_inputs,
    encoder_states,
    name="encoder_inference_model"
)


# ------------------------------------------------------------
# 15. BUILD DECODER INFERENCE MODEL
# ------------------------------------------------------------

decoder_state_input_h = keras.Input(
    shape=(latent_dim,),
    name="decoder_state_h"
)

decoder_state_input_c = keras.Input(
    shape=(latent_dim,),
    name="decoder_state_c"
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states = [
    state_h,
    state_c
]

decoder_outputs = decoder_dense(
    decoder_outputs
)

decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states,
    name="decoder_inference_model"
)


# ------------------------------------------------------------
# 16. REVERSE CHARACTER DICTIONARY
# ------------------------------------------------------------

reverse_target_char_index = {
    i: char
    for char, i in target_token_index.items()
}


# ------------------------------------------------------------
# 17. AUTOREGRESSIVE DECODING FUNCTION
# ------------------------------------------------------------

def decode_sequence(input_seq):

    # Encode the English sentence
    states_value = encoder_model.predict(
        input_seq,
        verbose=0
    )

    # Start-of-sequence character
    target_seq = np.zeros(
        (1, 1, num_decoder_tokens),
        dtype="float32"
    )

    target_seq[
        0,
        0,
        target_token_index["\t"]
    ] = 1.0

    decoded_sentence = ""

    for _ in range(max_decoder_seq_length):

        # Predict the next character
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        # Get the character with highest probability
        sampled_token_index = int(
            np.argmax(output_tokens[0, -1, :])
        )

        sampled_char = reverse_target_char_index[
            sampled_token_index
        ]

        # Stop when end-of-sequence is generated
        if sampled_char == "\n":
            break

        decoded_sentence += sampled_char

        # Prepare predicted character as next input
        target_seq = np.zeros(
            (1, 1, num_decoder_tokens),
            dtype="float32"
        )

        target_seq[
            0,
            0,
            sampled_token_index
        ] = 1.0

        # Update decoder states
        states_value = [h, c]

    return decoded_sentence


# ------------------------------------------------------------
# 18. TEST TRANSLATIONS
# ------------------------------------------------------------

print("\n========================================")
print("TEST TRANSLATIONS")
print("========================================")

num_tests = min(20, len(input_texts))

for seq_index in range(num_tests):

    input_seq = encoder_input_data[
        seq_index:seq_index + 1
    ]

    decoded_sentence = decode_sequence(
        input_seq
    )

    actual_french = target_texts[
        seq_index
    ][1:-1]

    print("\nEnglish :", input_texts[seq_index])
    print("Actual  :", actual_french)
    print("Predicted:", decoded_sentence)


# ------------------------------------------------------------
# 19. FINAL INFORMATION
# ------------------------------------------------------------

print("\n========================================")
print("PROBLEM 1 COMPLETED")
print("========================================")

print("Training samples:", len(input_texts))
print("Epochs used:", epochs)
print("Latent dimension:", latent_dim)
print("Encoder vocabulary:", num_encoder_tokens)
print("Decoder vocabulary:", num_decoder_tokens)

print("\nTraining model:")
print("s2s_model.keras")

print("\nInference models created:")
print("encoder_model")
print("decoder_model")

print("\nThe decoder uses autoregressive inference:")
print("previous prediction → next prediction")

Keras version: 3.13.2
Training samples: 10000
Epochs: 20
Latent dimension: 256

Dataset found: /root/.keras/datasets/fra-eng_extracted/fra.txt

===== DATA INFORMATION =====
Number of samples: 10000
Number of input characters: 70
Number of target characters: 93
Maximum encoder sequence length: 16
Maximum decoder sequence length: 59

===== TENSOR SHAPES =====
Encoder input: (10000, 16, 70)
Decoder input: (10000, 59, 93)
Decoder target: (10000, 59, 93)

===== MODEL SUMMARY =====


Model: "seq2seq_translation_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None, 70)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None, 93)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    334,848 │ encoder_inputs[0… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    358,400 │ decoder_inputs[0… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 93)  │     23,901 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 717,149 (2.74 MB)

 Trainable params: 717,149 (2.74 MB)

 Non-trainable params: 0 (0.00 B)


===== STARTING TRAINING =====
Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 55s 423ms/step - accuracy: 0.7180 - loss: 1.2846 - val_accuracy: 0.6908 - val_loss: 1.2286
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 424ms/step - accuracy: 0.7344 - loss: 0.9912 - val_accuracy: 0.7063 - val_loss: 1.0455
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 82s 421ms/step - accuracy: 0.7540 - loss: 0.8870 - val_accuracy: 0.7388 - val_loss: 0.9510
Epoch 4/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 54s 432ms/step - accuracy: 0.7792 - loss: 0.7917 - val_accuracy: 0.7599 - val_loss: 0.8514
Epoch 5/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 54s 431ms/step - accuracy: 0.7974 - loss: 0.7073 - val_accuracy: 0.7739 - val_loss: 0.7845
Epoch 6/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 53s 424ms/step - accuracy: 0.8068 - loss: 0.6645 - val_accuracy: 0.7765 - val_loss: 0.7535
Epoch 7/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 52s 419ms/step - accuracy: 0.8141 - loss: 0.6349 - val_accuracy: 0.7869 - val_loss: 0.7209
Epoch 8/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 53s 421ms/st

In [ ]:
## Problem 2

In [ ]:
# ============================================================
# PROBLEM 2 — PRETRAINED IMAGE CAPTIONING WITH BLIP
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL CURRENT TRANSFORMERS
# ------------------------------------------------------------

!pip -q install -U transformers


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

from pathlib import Path
import requests
import torch

from PIL import Image
from transformers import AutoProcessor, BlipForConditionalGeneration


# ------------------------------------------------------------
# 3. CHECK RUNTIME DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("========================================")
print("RUNTIME INFORMATION")
print("========================================")

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available — using CPU")


# ------------------------------------------------------------
# 4. LOAD PRETRAINED BLIP MODEL
# ------------------------------------------------------------

checkpoint = "Salesforce/blip-image-captioning-base"

print("\n========================================")
print("LOADING BLIP MODEL")
print("========================================")

print("Checkpoint:", checkpoint)

processor = AutoProcessor.from_pretrained(
    checkpoint
)

model = BlipForConditionalGeneration.from_pretrained(
    checkpoint
)

model = model.to(device)
model.eval()

print("BLIP model loaded successfully.")


# ------------------------------------------------------------
# 5. PREPARE THREE DIFFERENT IMAGES
# ------------------------------------------------------------
#
# These are public example images.
# The images are downloaded only once.
#
# Image 1: woman and dog
# Image 2: dog
# Image 3: outdoor scene
# ------------------------------------------------------------

image_dir = Path("seq2seq_images")
image_dir.mkdir(exist_ok=True)

image_urls = {
    "image_1_woman_dog.jpg":
        "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg",

    "image_2_dog.jpg":
        "https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png",

    "image_3_scene.jpg":
        "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/image_captioning.png"
}


# ------------------------------------------------------------
# 6. DOWNLOAD IMAGES
# ------------------------------------------------------------

print("\n========================================")
print("DOWNLOADING IMAGES")
print("========================================")

image_paths = []

for filename, url in image_urls.items():

    path = image_dir / filename

    if not path.exists():

        response = requests.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        with open(path, "wb") as f:
            f.write(response.content)

        print("Downloaded:", filename)

    else:
        print("Already cached:", filename)

    image_paths.append(path)


# ------------------------------------------------------------
# 7. GENERATE CAPTIONS
# ------------------------------------------------------------

print("\n========================================")
print("IMAGE CAPTIONING RESULTS")
print("========================================")

results = []

for image_path in image_paths:

    print("\n----------------------------------------")
    print("Image:", image_path.name)

    # Open image
    image = Image.open(image_path).convert("RGB")

    print("Image size:", image.size)

    # Preprocess image
    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    # Move tensors to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate caption
    with torch.inference_mode():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30
        )

    # Convert tokens back to text
    caption = processor.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    print("Generated caption:")
    print(caption)

    # Store result
    results.append(
        {
            "image": image_path.name,
            "caption": caption
        }
    )


# ------------------------------------------------------------
# 8. DISPLAY ALL RESULTS
# ------------------------------------------------------------

print("\n========================================")
print("SUMMARY")
print("========================================")

for result in results:

    print("\nImage:", result["image"])
    print("Caption:", result["caption"])


# ------------------------------------------------------------
# 9. MODEL INFORMATION FOR THE REPORT
# ------------------------------------------------------------

print("\n========================================")
print("MODEL INFORMATION")
print("========================================")

print("Model:", checkpoint)
print("Task: Image captioning")
print("Pretrained: Yes")
print("Training performed in this problem: No")
print("Runtime device:", device)

print("\nLicense: BSD-3-Clause")

print(
    "\nIntended use: pretrained image-captioning "
    "research and experimentation."
)

print(
    "\nImportant limitation: captions may be incomplete "
    "or inaccurate, especially for complex or unusual scenes."
)

print(
    "\nThe BLIP model card recommends evaluating "
    "accuracy, safety, and fairness before deployment."
)


# ------------------------------------------------------------
# 10. SHORT EVALUATION
# ------------------------------------------------------------

print("\n========================================")
print("EVALUATION GUIDE")
print("========================================")

print("""
For each image, compare the generated caption with
the actual image.

Check:

1. Did the model identify the main subject?
2. Did it identify the main action?
3. Did it identify the environment?
4. Did it miss important objects?
5. Did it incorrectly describe anything?
6. Is the caption specific or too general?
""")


print("\n========================================")
print("PROBLEM 2 COMPLETED")
print("========================================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 22.0 MB/s eta 0:00:00
RUNTIME INFORMATION
PyTorch version: 2.11.0+cpu
Device: cpu
GPU not available — using CPU

LOADING BLIP MODEL
Checkpoint: Salesforce/blip-image-captioning-base


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

BLIP model loaded successfully.

DOWNLOADING IMAGES
Downloaded: image_1_woman_dog.jpg
Downloaded: image_2_dog.jpg


HTTPError: 404 Client Error: Not Found for url: https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/image_captioning.png

In [ ]:
# ============================================================
# PROBLEM 2 — IMAGE CAPTIONING
# Continue from the already loaded model
# ============================================================

from pathlib import Path
import requests
from PIL import Image
from io import BytesIO
import torch

# Folder
image_dir = Path("seq2seq_images")
image_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Download a reliable third image
# ------------------------------------------------------------

image_3_path = image_dir / "image_3_scene.jpg"

image_3_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"

response = requests.get(image_3_url, timeout=30)
response.raise_for_status()

with open(image_3_path, "wb") as f:
    f.write(response.content)

print("Third image downloaded successfully:")
print(image_3_path)

# ------------------------------------------------------------
# 2. Check the three images
# ------------------------------------------------------------

image_paths = sorted(image_dir.glob("*"))

print("\nImages available:")
for path in image_paths:
    print("-", path)

# ------------------------------------------------------------
# 3. Generate captions
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("IMAGE CAPTIONING RESULTS")
print("=" * 60)

results = []

for image_path in image_paths:

    # Open image
    image = Image.open(image_path).convert("RGB")

    # Prepare input
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate caption
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30
        )

    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    results.append((image_path.name, caption))

    print(f"\nImage: {image_path.name}")
    print(f"Caption: {caption}")

# ------------------------------------------------------------
# 4. Device information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DEVICE INFORMATION")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("Device used:", device)
print("GPU available:", torch.cuda.is_available())

# ------------------------------------------------------------
# 5. Short evaluation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

for name, caption in results:
    print(f"\n{name}")
    print("Generated caption:", caption)
    print("Evaluation: Check whether the caption correctly identifies")
    print("the main subject and important elements of the image.")

Third image downloaded successfully:
seq2seq_images/image_3_scene.jpg

Images available:
- seq2seq_images/image_1_woman_dog.jpg
- seq2seq_images/image_2_dog.jpg
- seq2seq_images/image_3_scene.jpg

IMAGE CAPTIONING RESULTS

Image: image_1_woman_dog.jpg
Caption: a woman sitting on the beach with her dog

Image: image_2_dog.jpg
Caption: two birds are standing next to each other birds

Image: image_3_scene.jpg
Caption: a white dog sitting in the grass

DEVICE INFORMATION
PyTorch version: 2.11.0+cpu
Device used: cpu
GPU available: False

EVALUATION

image_1_woman_dog.jpg
Generated caption: a woman sitting on the beach with her dog
Evaluation: Check whether the caption correctly identifies
the main subject and important elements of the image.

image_2_dog.jpg
Generated caption: two birds are standing next to each other birds
Evaluation: Check whether the caption correctly identifies
the main subject and important elements of the image.

image_3_scene.jpg
Generated caption: a white dog sittin

In [ ]:
## Problem 3

In [ ]:
# ============================================================
# PROBLEM 3 — RUNNING IMAGE CAPTIONING WITH KERAS 3
# ============================================================

import keras
import tensorflow as tf
import numpy as np

print("=" * 70)
print("PROBLEM 3 — KERAS 3 IMAGE CAPTIONING INVESTIGATION")
print("=" * 70)

print("\n1. CURRENT ENVIRONMENT")
print("-" * 70)
print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# 2. APPROACH 1 — BUILD AND TRAIN WITH KERAS
# ------------------------------------------------------------

print("\n2. APPROACH 1 — BUILD AND TRAIN A KERAS IMAGE-CAPTIONING MODEL")
print("-" * 70)

print("""
A current Keras 3 image-captioning system can be built using Keras layers.

Typical architecture:

Image
  |
  v
CNN image encoder
  |
  v
Visual feature sequence
  |
  v
Transformer encoder
  |
  v
Transformer decoder
  |
  v
Dense + Softmax
  |
  v
Next-token probabilities

The CNN converts the image into visual features.
The Transformer encoder processes these visual features.
The Transformer decoder receives caption tokens and attends to
the encoded image features.
A Dense layer with softmax produces the probability of the next token.

During training, teacher forcing is used:
decoder input  = <start> + caption[:-1]
target          = caption

The model can then be saved directly in the current Keras format:

    model.save("image_captioning.keras")

This is a genuine Keras implementation because the architecture
and parameters are created and trained using Keras.
""")

# ------------------------------------------------------------
# 3. APPROACH 2 — PORT PRETRAINED PYTORCH PARAMETERS
# ------------------------------------------------------------

print("\n3. APPROACH 2 — PORT PRETRAINED PYTORCH PARAMETERS")
print("-" * 70)

print("""
A PyTorch checkpoint cannot simply be loaded as a .keras model.

The reason is that .keras stores a Keras model's architecture,
configuration, variables, and training information. A PyTorch
checkpoint contains parameters belonging to a PyTorch architecture.

Therefore, weight conversion requires an equivalent Keras model.
""")

# ------------------------------------------------------------
# 4. EXACT CORRESPONDENCE
# ------------------------------------------------------------

print("\n4. REQUIRED WEIGHT-CONVERSION CHECKS")
print("-" * 70)

checks = [
    ("Architecture",
     "Every PyTorch layer must have an exactly corresponding Keras layer."),

    ("Layer names / mapping",
     "Create an explicit source-layer -> destination-layer mapping."),

    ("Input preprocessing",
     "Use the same image resizing, cropping, normalization and color convention."),

    ("Tokenizer",
     "Use the same vocabulary, token IDs, special tokens and tokenization rules."),

    ("Special tokens",
     "Verify <start>, <end>, padding and unknown-token IDs exactly."),

    ("Generation",
     "Use the same maximum length, decoding strategy, temperature, beam size and stopping rule."),

    ("Dense kernels",
     "Check tensor layout because PyTorch Linear and Keras Dense may store weights differently."),

    ("Convolution kernels",
     "Check PyTorch Conv2D layout against Keras Conv2D layout and transpose when necessary."),

    ("Layer normalization",
     "Match epsilon, scale/gamma and bias/beta parameters."),

    ("Bias",
     "Verify whether every source layer uses a bias and whether the destination layer does too."),

    ("Activation",
     "Use the same activation function and numerical definition."),

    ("Layer ordering",
     "Verify operations occur in exactly the same order."),

    ("Array assignment",
     "Convert each parameter to the required shape and assign it to the Keras variable."),

    ("Intermediate tensors",
     "Compare activations from both models using identical inputs."),

    ("Final logits",
     "Compare the final token logits before applying the decoding procedure."),

    ("Numerical tolerance",
     "Use a small numerical tolerance because floating-point implementations can differ."),

    ("Caption generation",
     "Run both models with identical inputs and generation settings and compare captions."),

    ("Final model",
     "Save only after the converted Keras model has been verified.")
]

for name, explanation in checks:
    print(f"\n{name}:")
    print("  " + explanation)

# ------------------------------------------------------------
# 5. IMPORTANT TENSOR LAYOUT EXAMPLES
# ------------------------------------------------------------

print("\n5. TENSOR LAYOUT DIFFERENCES")
print("-" * 70)

print("""
Example: Dense / Linear layers

PyTorch Linear commonly stores:

    [out_features, in_features]

A Keras Dense kernel is commonly:

    [in_features, out_features]

Therefore a transpose may be required:

    keras_kernel = pytorch_weight.T


Example: convolution

Convolution kernels may also use different dimension ordering
between frameworks. The exact source and destination shapes must
be checked before assignment.

The conversion must NEVER assume that equal numbers of parameters
mean equal tensor layouts.
""")

# ------------------------------------------------------------
# 6. NUMERICAL VERIFICATION
# ------------------------------------------------------------

print("\n6. NUMERICAL VERIFICATION PLAN")
print("-" * 70)

print("""
The safest conversion procedure is:

1. Prepare exactly the same input image.
2. Apply exactly the same preprocessing.
3. Run the PyTorch model.
4. Run the converted Keras model.
5. Compare intermediate tensors layer by layer.
6. Compare the final logits.
7. Use np.testing.assert_allclose() with an appropriate tolerance.
8. Compare the generated captions using identical decoding settings.
9. Save the verified model as:

       model.save("verified_image_captioning.keras")

A successful conversion requires agreement within an appropriate
floating-point tolerance, not merely successful execution.
""")

# ------------------------------------------------------------
# 7. ARCHITECTURE MISMATCH
# ------------------------------------------------------------

print("\n7. WHAT IF THE ARCHITECTURES DO NOT MATCH?")
print("-" * 70)

print("""
Direct weight conversion is NOT valid if the architectures do not
match exactly.

Three different situations must be distinguished:

A. REIMPLEMENTATION
   Build a similar architecture in Keras and train it.
   The original PyTorch weights are not directly transferred.

B. KNOWLEDGE DISTILLATION
   Use the pretrained PyTorch model as a teacher and train a
   Keras student model to reproduce its outputs.
   This is not direct weight conversion.

C. RETRAINING
   Build the Keras architecture and train it using the dataset.
   Again, the original checkpoint is not directly converted.

Therefore, a different architecture cannot be described as a
successful PyTorch-to-Keras weight conversion.
""")

# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n8. FINAL SUMMARY")
print("-" * 70)

print("""
Keras 3 can implement modern image-captioning systems using
CNN-based visual encoders and Transformer encoder-decoder layers.

There are therefore two valid approaches:

1. Build and train an image-captioning architecture directly with Keras.
2. Convert pretrained PyTorch weights only when the Keras architecture,
   preprocessing, tokenizer, parameter shapes, tensor layouts and
   numerical outputs have been verified to be equivalent.

A PyTorch checkpoint cannot simply be renamed or loaded as a .keras file.

For a reliable conversion, layer-by-layer parameter mapping,
tensor-layout conversion, intermediate activation comparison,
final-logit comparison, numerical tolerance checks and final
caption-generation comparison are required.

Only after these checks should the model be saved in the current
.keras format.
""")

print("=" * 70)
print("PROBLEM 3 COMPLETE")
print("=" * 70)

PROBLEM 3 — KERAS 3 IMAGE CAPTIONING INVESTIGATION

1. CURRENT ENVIRONMENT
----------------------------------------------------------------------
Keras version: 3.13.2
TensorFlow version: 2.20.0

2. APPROACH 1 — BUILD AND TRAIN A KERAS IMAGE-CAPTIONING MODEL
----------------------------------------------------------------------

A current Keras 3 image-captioning system can be built using Keras layers.

Typical architecture:

Image
  |
  v
CNN image encoder
  |
  v
Visual feature sequence
  |
  v
Transformer encoder
  |
  v
Transformer decoder
  |
  v
Dense + Softmax
  |
  v
Next-token probabilities

The CNN converts the image into visual features.
The Transformer encoder processes these visual features.
The Transformer decoder receives caption tokens and attends to
the encoded image features.
A Dense layer with softmax produces the probability of the next token.

During training, teacher forcing is used:
decoder input  = <start> + caption[:-1]
target          = caption

The model can 

In [ ]:
## Problem 4

In [ ]:
# ============================================================
# PROBLEM 4 — SMALL IMAGE-CAPTIONING MODEL WITH KERAS 3
# CORRECTED VERSION
# ============================================================

import keras
from keras import layers
import tensorflow as tf
import numpy as np

print("=" * 70)
print("PROBLEM 4 — KERAS 3 IMAGE CAPTIONING")
print("=" * 70)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

IMAGE_HEIGHT = 64
IMAGE_WIDTH = 64
IMAGE_CHANNELS = 3

VOCAB_SIZE = 1000

PAD_ID = 0
START_ID = 1
END_ID = 2

MAX_LENGTH = 12

EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

BATCH_SIZE = 2

print("\nConfiguration:")
print("Image input:", (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))
print("Vocabulary size:", VOCAB_SIZE)
print("PAD token ID:", PAD_ID)
print("START token ID:", START_ID)
print("END token ID:", END_ID)
print("Maximum caption length:", MAX_LENGTH)
print("Embedding dimension:", EMBED_DIM)


# ============================================================
# 2. IMAGE ENCODER
# ============================================================

class ImageEncoder(layers.Layer):

    def __init__(self, embed_dim):
        super().__init__()

        self.cnn = keras.Sequential([
            layers.Conv2D(
                32,
                3,
                activation="relu",
                padding="same"
            ),

            layers.MaxPooling2D(2),

            layers.Conv2D(
                64,
                3,
                activation="relu",
                padding="same"
            ),

            layers.MaxPooling2D(2),

            layers.Conv2D(
                embed_dim,
                3,
                activation="relu",
                padding="same"
            )
        ])

        self.projection = layers.Dense(embed_dim)

    def call(self, images):

        # CNN feature map
        x = self.cnn(images)

        # Convert spatial feature map into a sequence
        # Example:
        # (batch, 16, 16, 64)
        # ->
        # (batch, 256, 64)

        batch_size = tf.shape(x)[0]

        x = tf.reshape(
            x,
            [batch_size, -1, x.shape[-1]]
        )

        x = self.projection(x)

        return x


# ============================================================
# 3. TRANSFORMER DECODER
# ============================================================

class TransformerDecoder(layers.Layer):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_dim
    ):
        super().__init__()

        self.embedding = layers.Embedding(
            vocab_size,
            embed_dim
        )

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.cross_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ffn = keras.Sequential([
            layers.Dense(
                ff_dim,
                activation="relu"
            ),

            layers.Dense(embed_dim)
        ])

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.norm3 = layers.LayerNormalization()

        self.output_layer = layers.Dense(
            vocab_size
        )

    def call(
        self,
        token_ids,
        image_features,
        training=False
    ):

        # ----------------------------------------------------
        # Token embedding
        # ----------------------------------------------------

        x = self.embedding(token_ids)

        # Current sequence length
        seq_len = tf.shape(token_ids)[1]

        # ----------------------------------------------------
        # Causal mask
        # ----------------------------------------------------
        #
        # Token i can only attend to tokens <= i.
        #

        causal_mask = tf.linalg.band_part(
            tf.ones(
                (seq_len, seq_len),
                dtype=tf.bool
            ),
            -1,
            0
        )

        # ----------------------------------------------------
        # Padding mask
        # ----------------------------------------------------

        padding_mask = tf.not_equal(
            token_ids,
            PAD_ID
        )

        # Shape:
        # (batch, seq_len)
        #
        # Convert to:
        # (batch, seq_len, seq_len)

        padding_mask = tf.expand_dims(
            padding_mask,
            axis=1
        )

        padding_mask = tf.broadcast_to(
            padding_mask,
            [
                tf.shape(token_ids)[0],
                seq_len,
                seq_len
            ]
        )

        # Combine padding + causal masks
        attention_mask = (
            padding_mask
            & causal_mask
        )

        # ----------------------------------------------------
        # 1. MASKED SELF-ATTENTION
        # ----------------------------------------------------

        self_attention_output = self.self_attention(
            query=x,
            key=x,
            value=x,
            attention_mask=attention_mask,
            training=training
        )

        x = self.norm1(
            x + self_attention_output
        )

        # ----------------------------------------------------
        # 2. CROSS-ATTENTION
        # ----------------------------------------------------

        cross_attention_output = self.cross_attention(
            query=x,
            key=image_features,
            value=image_features,
            training=training
        )

        x = self.norm2(
            x + cross_attention_output
        )

        # ----------------------------------------------------
        # 3. FEED-FORWARD NETWORK
        # ----------------------------------------------------

        ffn_output = self.ffn(
            x,
            training=training
        )

        x = self.norm3(
            x + ffn_output
        )

        # ----------------------------------------------------
        # 4. VOCABULARY LOGITS
        # ----------------------------------------------------

        logits = self.output_layer(x)

        return logits


# ============================================================
# 4. COMPLETE IMAGE-CAPTIONING MODEL
# ============================================================

class ImageCaptioningModel(keras.Model):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ff_dim
    ):
        super().__init__()

        self.image_encoder = ImageEncoder(
            embed_dim
        )

        self.decoder = TransformerDecoder(
            vocab_size,
            embed_dim,
            num_heads,
            ff_dim
        )

    def call(
        self,
        inputs,
        training=False
    ):

        images = inputs["images"]
        token_ids = inputs["token_ids"]

        # Image -> visual features
        image_features = self.image_encoder(
            images
        )

        # Visual features -> decoder
        logits = self.decoder(
            token_ids,
            image_features,
            training=training
        )

        return logits


# ============================================================
# 5. EXPLICIT KERAS INPUTS
# ============================================================

model = ImageCaptioningModel(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM
)

image_input = keras.Input(
    shape=(
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        IMAGE_CHANNELS
    ),
    name="images"
)

# IMPORTANT:
# None allows the decoder sequence length to be 11 during
# teacher forcing instead of forcing it to be 12.

token_input = keras.Input(
    shape=(None,),
    dtype="int32",
    name="token_ids"
)

outputs = model(
    {
        "images": image_input,
        "token_ids": token_input
    }
)

keras_model = keras.Model(
    inputs={
        "images": image_input,
        "token_ids": token_input
    },
    outputs=outputs
)


# ============================================================
# 6. CREATE SMALL DUMMY BATCH
# ============================================================

dummy_images = np.random.random(
    (
        BATCH_SIZE,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        IMAGE_CHANNELS
    )
).astype("float32")


# Example captions:
#
# <START> woman is walking <END> <PAD> ...
#
dummy_token_ids = np.array([
    [
        START_ID,
        10,
        25,
        37,
        45,
        END_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID
    ],

    [
        START_ID,
        15,
        20,
        30,
        END_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID,
        PAD_ID
    ]

], dtype="int32")


# ============================================================
# 7. TEACHER FORCING
# ============================================================

# Decoder input:
#
# <START> woman is walking
#
# Target:
#
# woman is walking <END>

decoder_input = dummy_token_ids[:, :-1]

target_tokens = dummy_token_ids[:, 1:]

print("\nTraining tensors:")
print("Full caption shape:", dummy_token_ids.shape)
print("Decoder input shape:", decoder_input.shape)
print("Target shape:", target_tokens.shape)

print("\nExample decoder input:")
print(decoder_input[0])

print("\nExample target:")
print(target_tokens[0])


# ============================================================
# 8. FORWARD PASS
# ============================================================

logits = keras_model(
    {
        "images": dummy_images,
        "token_ids": decoder_input
    },
    training=False
)

print("\nForward pass:")
print("Logits shape:", logits.shape)

expected_shape = (
    BATCH_SIZE,
    MAX_LENGTH - 1,
    VOCAB_SIZE
)

print("Expected shape:", expected_shape)

assert logits.shape == expected_shape

print("\nSUCCESS: Forward pass has the expected shape.")


# ============================================================
# 9. VISUAL FEATURE SHAPE
# ============================================================

visual_features = model.image_encoder(
    dummy_images
)

print("\nVisual features:")
print("Visual feature shape:", visual_features.shape)

print("""
The CNN converts each image into a sequence of visual features.
The Transformer decoder receives these features through
cross-attention.
""")


# ============================================================
# 10. TRAINING LOSS
# ============================================================

loss_function = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

loss = loss_function(
    target_tokens,
    logits
)

print("Initial training loss:", float(loss))


# ============================================================
# 11. SEPARATE AUTOREGRESSIVE INFERENCE
# ============================================================

def generate_caption(
    image,
    model,
    max_length=MAX_LENGTH
):

    # Start with <START>
    generated_tokens = [
        START_ID
    ]

    for step in range(max_length - 1):

        token_tensor = tf.constant(
            [generated_tokens],
            dtype=tf.int32
        )

        # Image -> visual features
        image_features = model.image_encoder(
            tf.expand_dims(image, axis=0)
        )

        # Decoder
        logits = model.decoder(
            token_tensor,
            image_features,
            training=False
        )

        # Last predicted timestep
        next_token_logits = logits[:, -1, :]

        # Greedy decoding
        next_token = int(
            tf.argmax(
                next_token_logits,
                axis=-1
            )[0]
        )

        generated_tokens.append(
            next_token
        )

        # Stop at <END>
        if next_token == END_ID:
            break

    return generated_tokens


# ============================================================
# 12. TEST AUTOREGRESSIVE GENERATION
# ============================================================

generated_tokens = generate_caption(
    dummy_images[0],
    model
)

print("\nAutoregressive inference:")
print("Generated token IDs:")
print(generated_tokens)

print("""
Inference procedure:

1. Start with <START>.
2. Predict the next token.
3. Append the predicted token.
4. Feed the updated sequence back to the decoder.
5. Continue until <END> or MAX_LENGTH.
""")


# ============================================================
# 13. MODEL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MODEL SUMMARY")
print("=" * 70)

keras_model.summary()


# ============================================================
# 14. FINAL CHECKS
# ============================================================

print("\n" + "=" * 70)
print("PROBLEM 4 CHECKLIST")
print("=" * 70)

print("[✓] Image input shape:", dummy_images.shape)
print("[✓] Visual feature shape:", visual_features.shape)
print("[✓] Token IDs")
print("[✓] Padding token ID:", PAD_ID)
print("[✓] Start token ID:", START_ID)
print("[✓] End token ID:", END_ID)
print("[✓] Image encoder -> decoder")
print("[✓] Causal self-attention mask")
print("[✓] Padding mask")
print("[✓] Training logits:", logits.shape)
print("[✓] Target shape:", target_tokens.shape)
print("[✓] Forward pass confirmed")
print("[✓] Separate autoregressive inference loop")
print("[✓] No BLIP weight compatibility claimed")

print("\nPROBLEM 4 COMPLETE")

PROBLEM 4 — KERAS 3 IMAGE CAPTIONING

Configuration:
Image input: (64, 64, 3)
Vocabulary size: 1000
PAD token ID: 0
START token ID: 1
END token ID: 2
Maximum caption length: 12
Embedding dimension: 64

Training tensors:
Full caption shape: (2, 12)
Decoder input shape: (2, 11)
Target shape: (2, 11)

Example decoder input:
[ 1 10 25 37 45  2  0  0  0  0  0]

Example target:
[10 25 37 45  2  0  0  0  0  0  0]

Forward pass:
Logits shape: (2, 11, 1000)
Expected shape: (2, 11, 1000)

SUCCESS: Forward pass has the expected shape.

Visual features:
Visual feature shape: (2, 256, 64)

The CNN converts each image into a sequence of visual features.
The Transformer decoder receives these features through
cross-attention.

Initial training loss: 6.5871429443359375

Autoregressive inference:
Generated token IDs:
[1, 944, 669, 272, 245, 267, 316, 15, 865, 649, 944, 669]

Inference procedure:

1. Start with <START>.
2. Predict the next token.
3. Append the predicted token.
4. Feed the updated sequen

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer) │ (None, 64, 64, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_captioning_m… │ (None, None,      │    239,720 │ images[0][0],     │
│ (ImageCaptioningMo… │ 1000)             │            │ token_ids[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 239,720 (936.41 KB)

 Trainable params: 239,720 (936.41 KB)

 Non-trainable params: 0 (0.00 B)


PROBLEM 4 CHECKLIST
[✓] Image input shape: (2, 64, 64, 3)
[✓] Visual feature shape: (2, 256, 64)
[✓] Token IDs
[✓] Padding token ID: 0
[✓] Start token ID: 1
[✓] End token ID: 2
[✓] Image encoder -> decoder
[✓] Causal self-attention mask
[✓] Padding mask
[✓] Training logits: (2, 11, 1000)
[✓] Target shape: (2, 11)
[✓] Forward pass confirmed
[✓] Separate autoregressive inference loop
[✓] No BLIP weight compatibility claimed

PROBLEM 4 COMPLETE


In [ ]:
## Problem 5

In [ ]:
# Problem 5 — Developmental Survey

## 1. Translating between Japanese and English

Adapting the model from Problem 1 to Japanese-English translation requires several additional steps. Simply replacing the English-French training sentences with Japanese-English sentences is insufficient because Japanese has different writing, tokenization, and sequence-processing requirements.

### Unicode normalization

Japanese text should first be normalized using Unicode normalization such as **NFKC**. This reduces inconsistencies between different Unicode representations of visually similar characters and makes the training data more consistent.

### Japanese-capable subword tokenizer

The character-level approach used in Problem 1 can be replaced or extended with a **Japanese-capable subword tokenizer**, such as SentencePiece or BPE. Japanese normally does not use spaces between words, so simple whitespace-based word tokenization is inappropriate.

A subword tokenizer divides sentences into reusable units and helps control vocabulary size while allowing the model to represent unknown or rare words through smaller subword units.

### Licensed parallel corpus

The model requires a **Japanese-English parallel corpus**, where each Japanese sentence is aligned with an English translation. The dataset must have a license that permits its intended research or educational use.

The data should also be cleaned to remove duplicates, corrupted sentences, incorrect alignments, and inappropriate examples.

### Special tokens

The translation system should define special tokens such as:

* `<PAD>` — padding
* `<UNK>` — unknown token
* `<START>` — beginning of target sequence
* `<END>` — end of target sequence

These tokens are necessary for sequence preparation and autoregressive decoding.

### Padding and masking

Japanese and English sentences have different lengths. Sentences in the same batch therefore need to be padded to a common length using `<PAD>`.

Padding positions should be **masked** so that the model does not treat artificial padding as meaningful information.

The decoder also requires **causal masking** during training. This prevents the decoder from seeing future target tokens when predicting the current token.

### Train/validation/test separation

The parallel corpus should be divided into:

* **Training set:** used to learn the model parameters.
* **Validation set:** used for model selection and hyperparameter tuning.
* **Test set:** used for final evaluation.

The test set should remain unseen during training and model selection.

### Evaluation

Translation quality can be evaluated using **BLEU** and **chrF**.

BLEU compares generated translations with reference translations using n-gram precision. chrF evaluates character n-gram similarity and can be useful when word segmentation differs between languages.

Human evaluation should also be considered because automatic metrics cannot completely measure meaning, fluency, adequacy, or naturalness.

Therefore, adapting Problem 1 requires changes to the preprocessing pipeline, tokenizer, vocabulary, sequence preparation, masking, dataset organization, and evaluation procedure. Merely replacing the training text would not address these differences.

---

# 2. Advanced machine-translation methods

## Attention-based encoder-decoder models

Problem 1 uses a fixed-state LSTM encoder-decoder. The encoder processes the source sentence and produces a final hidden representation that is passed to the decoder.

This creates an information bottleneck, especially for long sentences.

An **attention mechanism** allows the decoder to access different encoder hidden states while generating each target token. Therefore, the decoder can focus on the most relevant parts of the source sentence at each step.

Attention improves the ability to handle longer sequences compared with a basic fixed-state encoder-decoder.

## Transformers

The **Transformer** was introduced in *Attention Is All You Need*, published on June 12, 2017. It replaced recurrent processing with attention-based mechanisms and introduced self-attention, multi-head attention, and encoder-decoder attention. The original paper reported improved translation performance and more parallelizable training compared with recurrent approaches.

Compared with the fixed-state LSTM from Problem 1, a Transformer:

* does not depend on recurrent processing;
* can model relationships between distant tokens using attention;
* allows more parallel computation during training;
* uses self-attention in the encoder and decoder;
* uses cross-attention between source and target representations.

Thus, the Transformer is substantially more powerful for modern machine translation than the small fixed-state LSTM implemented in Problem 1.

## Pretrained multilingual encoder-decoder models

Another important development is the use of **pretrained multilingual encoder-decoder models**.

Instead of training a Japanese-English model completely from scratch, a pretrained multilingual model can be trained on many languages and then used or adapted for a particular translation direction.

Examples include multilingual translation models such as **NLLB-200**, which was introduced by Meta in 2022. Such models can provide translation capabilities across many languages and can reduce the amount of task-specific training required.

The disadvantage is that these models are much larger and more computationally demanding than the small LSTM used in Problem 1.

## Decoding methods

During inference, the model generates the target sentence token by token.

### Greedy decoding

Greedy decoding selects the highest-probability token at every step.

It is simple and fast, but a locally optimal choice may not produce the best complete translation.

### Beam search

**Beam search** maintains several candidate sequences during decoding. For example, with a beam size of 4, four promising partial translations can be explored instead of only one.

This can produce better translations than greedy decoding but requires more computation.

### Length penalty

Beam search can favor short sequences because of how sequence probabilities are accumulated. A **length penalty** modifies the score according to the generated sequence length. Current Hugging Face Transformers documentation describes `length_penalty` as an exponential penalty applied to beam-based sequence scoring.

### Comparison

| Method                        | Main idea                                   | Compared with Problem 1                             |
| ----------------------------- | ------------------------------------------- | --------------------------------------------------- |
| Fixed-state LSTM              | Source sentence compressed into final state | Simple but has an information bottleneck            |
| Attention LSTM                | Decoder accesses encoder states             | Better for long sequences                           |
| Transformer                   | Self-attention and cross-attention          | More powerful and parallelizable                    |
| Pretrained multilingual model | Uses knowledge learned from many languages  | Reduces need for training from scratch              |
| Greedy decoding               | Select best token at each step              | Fast but can be suboptimal                          |
| Beam search                   | Maintains multiple candidate sequences      | Usually explores better alternatives but costs more |

---

# 3. Generating images from text

Text-to-image generation performs the opposite task from image captioning.

In **image captioning**:

**Image → Text**

The model receives visual information and generates a textual description.

In **text-to-image generation**:

**Text → Image**

The text prompt is converted into a representation that conditions an image-generation model.

Modern systems commonly use **diffusion models**. For example, the Stable Diffusion pipeline uses a text encoder to condition a diffusion-based image-generation process. The current Hugging Face Diffusers documentation describes a pipeline containing a text encoder, tokenizer, VAE, denoising U-Net, scheduler, and safety checker.

Therefore, the conditioning direction is fundamentally different:

* Captioning uses **visual features to generate language**.
* Text-to-image uses **language representations to guide image generation**.

## Evaluation limitations

Text-to-image generation does not have one unique correct output for a prompt. Therefore, evaluation is more difficult than evaluating a classification model.

Possible evaluation criteria include:

* alignment between the prompt and generated image;
* image quality;
* diversity;
* human judgment;
* similarity between text and image representations.

However, automatic metrics cannot completely determine whether an image is realistic, useful, culturally appropriate, or faithful to the intended meaning.

## Dataset and model licenses

The license of both the **training dataset** and the **model** must be considered before using a text-to-image system.

For example, some diffusion models were trained using large web-derived datasets, and their model cards specify particular restrictions and risks. Therefore, a model should not be assumed to be unrestricted simply because its weights are publicly available.

## Privacy

Text-to-image systems can create privacy concerns when users include personal or confidential information in prompts. Organizations should avoid sending sensitive information to external generation services unless appropriate privacy protections are in place.

## Bias

Generative models learn patterns from their training data. If the training data contains demographic, cultural, or gender biases, generated images may reproduce or amplify those biases.

For example, some model documentation reports limitations associated with under-representation of non-English communities and cultural groups.

## Misuse risks

Text-to-image models can be misused to create deceptive or harmful content, including manipulated representations of real people, inappropriate content, or misleading images.

Hugging Face's current Diffusers ethical guidance specifically discusses risks including copyright concerns, deepfake exploitation, non-consensual impersonation, and social bias.

Safety mechanisms can reduce some risks. For example, Diffusers documentation describes safety checking for certain Stable Diffusion pipelines and recommends keeping appropriate safety mechanisms enabled in public-facing applications.

## Practical investigation

A large text-to-image model does not need to be run for this investigation because the assignment specifically states that large models should only be run when necessary and when they fit the available Colab resources.

Similarly, a pretrained Japanese-English translation model can be investigated conceptually without retraining the LSTM from Problem 1. The main purpose of this part is to understand how modern pretrained models, tokenization, attention, and decoding differ from the simple fixed-state LSTM architecture.

---

# Conclusion

Adapting Problem 1 to Japanese-English translation requires substantial additional work. Unicode normalization, Japanese-capable subword tokenization, a properly licensed parallel corpus, special tokens, padding, masking, dataset separation, and appropriate evaluation are all necessary.

Modern machine translation has progressed from fixed-state LSTM encoder-decoder models toward attention-based models, Transformers, and large pretrained multilingual encoder-decoder models. Beam search and length penalties further improve the decoding process.

Text-to-image generation represents a different generative direction from image captioning because text representations condition an image-generation process rather than visual representations conditioning a language decoder. Its development also requires consideration of evaluation limitations, licensing, privacy, bias, and potential misuse.

### Time-sensitive sources

* Vaswani et al., *Attention Is All You Need*, published June 12, 2017.
* Hugging Face Transformers generation documentation, current documentation consulted September 2026, for beam search and length-penalty behavior.
* Hugging Face Diffusers documentation, current documentation consulted September 2026, for text-to-image pipelines and safety mechanisms.
